In [ ]:
from pathlib import Path
import shutil
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

COMP_DIR = Path("/kaggle/input/competitions/amia-public-challenge-2026")
WORK_DIR = Path("/kaggle/working/yolo_data")
RUN_DIR = Path("/kaggle/working/runs")
OUT_CSV = Path("/kaggle/working/submission.csv")
TRAIN_IMG_DIR = COMP_DIR / "train" / "train"
TEST_IMG_DIR = COMP_DIR / "test" / "test"
TRAIN_CSV = COMP_DIR / "train.csv"
IMG_SIZE_CSV = COMP_DIR / "img_size.csv"
SAMPLE_SUB = COMP_DIR / "sample_submission.csv"
RUN_NAME = "yolo26l-baseline"
MODEL = "yolo26l.pt"
EPOCHS = 150
IMG_SIZE = 640
BATCH = 8
CONF = 0.05
IOU = 0.40
NO_FINDING = 14
NC = 14

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
def key(x):
    return Path(str(x)).stem

In [ ]:
def make_dataset():
    train = pd.read_csv(TRAIN_CSV)
    sizes = pd.read_csv(IMG_SIZE_CSV)
    size_lookup = {
        key(r.image_id): (float(r.dim0), float(r.dim1))
        for r in sizes.itertuples(index=False)
    }
    train["image_key"] = train["image_id"].map(key)
    train = train[train["class_id"] != NO_FINDING].copy()
    image_ids = sorted([p.stem for p in TRAIN_IMG_DIR.glob("*.png")])
    train_ids, val_ids = train_test_split(image_ids, test_size=0.15, random_state=42)
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    for split, ids in [("train", train_ids), ("val", val_ids)]:
        (WORK_DIR / "images" / split).mkdir(parents=True)
        (WORK_DIR / "labels" / split).mkdir(parents=True)
        for image_id in ids:
            src = TRAIN_IMG_DIR / f"{image_id}.png"
            dst = WORK_DIR / "images" / split / src.name
            shutil.copy2(src, dst)
            orig_h, orig_w = size_lookup[image_id]
            rows = train[train["image_key"] == image_id]
            lines = []
            for r in rows.itertuples(index=False):
                x1 = max(0, min(float(r.x_min), orig_w - 1))
                y1 = max(0, min(float(r.y_min), orig_h - 1))
                x2 = max(0, min(float(r.x_max), orig_w - 1))
                y2 = max(0, min(float(r.y_max), orig_h - 1))
                if x2 <= x1 or y2 <= y1:
                    continue
                xc = ((x1 + x2) / 2) / orig_w
                yc = ((y1 + y2) / 2) / orig_h
                bw = (x2 - x1) / orig_w
                bh = (y2 - y1) / orig_h
                lines.append(f"{int(r.class_id)} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
            (WORK_DIR / "labels" / split / f"{image_id}.txt").write_text("\n".join(lines))
    yaml_path = WORK_DIR / "data.yaml"
    yaml_path.write_text(
        f"path: {WORK_DIR}\n"
        "train: images/train\n"
        "val: images/val\n"
        "names:\n"
        + "\n".join([f"  {i}: class_{i}" for i in range(NC)])
        + "\n"
    )
    return yaml_path, size_lookup

In [ ]:
def train_model(yaml_path):
    model = YOLO(MODEL)
    results = model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH*2,
        project=str(RUN_DIR),
        name=RUN_NAME,
        exist_ok=True,
        patience=15,
        device=[0,1],
        workers=4,
        plots=True,
    )
    return YOLO(str(Path(RUN_DIR / RUN_NAME / "weights" / "best.pt")))

In [ ]:
def make_submission(model, size_lookup):
    sample = pd.read_csv(SAMPLE_SUB)
    id_col = sample.columns[0]
    rows = []
    for raw_id in sample[id_col].astype(str):
        image_id = key(raw_id)
        img_path = TEST_IMG_DIR / f"{image_id}.png"
        img_w, img_h = Image.open(img_path).size
        orig_h, orig_w = size_lookup[image_id]
        pred = model.predict(
            str(img_path),
            imgsz=IMG_SIZE,
            conf=CONF,
            iou=IOU,
            verbose=False,
        )[0]
        tokens = []
        if pred.boxes is not None:
            boxes = pred.boxes.xyxy.cpu().numpy()
            scores = pred.boxes.conf.cpu().numpy()
            classes = pred.boxes.cls.cpu().numpy().astype(int)
            for box, score, cls in zip(boxes, scores, classes):
                x1, y1, x2, y2 = box
                x1 *= orig_w / img_w
                x2 *= orig_w / img_w
                y1 *= orig_h / img_h
                y2 *= orig_h / img_h
                tokens += [
                    str(cls),
                    f"{score:.6f}",
                    f"{x1:.1f}",
                    f"{y1:.1f}",
                    f"{x2:.1f}",
                    f"{y2:.1f}",
                ]
        pred_string = " ".join(tokens) if tokens else f"{NO_FINDING} 1 0 0 1 1"
        rows.append({
            id_col: raw_id,
            "PredictionString": pred_string,
        })
    pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
    print("Saved:", OUT_CSV)

In [8]:
yaml_path, size_lookup = make_dataset()
model = train_model(yaml_path)
make_submission(model, size_lookup)

Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_data/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26l-baseli